## PrimeKG — Harvard Medical Data Knowledge Graph as a Test case


In [ ]:
import numpy as np
import pandas as pd
from train import Train

e = Train()

In [ ]:
import subprocess
subprocess.run(['wget', '-O', 'kg.csv', 
    'https://dataverse.harvard.edu/api/access/datafile/6180620'], check=True)

In [ ]:
KEEP = {
    'indication', 'contraindication', 'off-label use',
    'drug_protein', 'disease_protein', 'disease_phenotype_positive'
}

chunks = []
for chunk in pd.read_csv('kg.csv', low_memory=False, chunksize=50000):
    chunks.append(chunk[chunk['relation'].isin(KEEP)])

df = pd.concat(chunks, ignore_index=True)
print(f"{len(df):,} edges")
print(df['relation'].value_counts())

In [ ]:
SEED = "Alzheimer"

seed_mask = df['x_name'].str.contains(SEED, case=False, na=False) | \
            df['y_name'].str.contains(SEED, case=False, na=False)

seed_ids = set(df[seed_mask]['x_id']) | set(df[seed_mask]['y_id'])

sub = df[df['x_id'].isin(seed_ids) & df['y_id'].isin(seed_ids)]
print(f"{len(sub):,} edges, {len(set(sub['x_id']) | set(sub['y_id'])):,} nodes")
print(sub['relation'].value_counts())

In [ ]:
all_ids     = pd.concat([sub['x_id'], sub['y_id']]).unique()
ent_to_idx  = {eid: i for i, eid in enumerate(all_ids)}
idx_to_name = {}
for _, row in sub[['x_id','x_name']].drop_duplicates().iterrows():
    idx_to_name[ent_to_idx[row['x_id']]] = row['x_name']
for _, row in sub[['y_id','y_name']].drop_duplicates().iterrows():
    idx_to_name[ent_to_idx[row['y_id']]] = row['y_name']

N = len(ent_to_idx)
relations = sub['relation'].unique().tolist()

R_dict = {}
for rel in relations:
    R = np.zeros((N, N), dtype=np.float32)
    for _, row in sub[sub['relation'] == rel].iterrows():
        R[ent_to_idx[row['x_id']], ent_to_idx[row['y_id']]] = 1.0
    R_dict[rel] = R
    print(f"  {rel}: {int(R.sum())} facts")

name_to_idx = {v: k for k, v in idx_to_name.items()}

In [ ]:
# Train
emb, EmbR = e.LearnEmbedPC(R_dict)

In [ ]:
def query(question, entity, *EmbRs, top_k=5, temp=0.05, semiring='fuzzy'):
    if isinstance(entity, str):
        if entity not in name_to_idx:
            print(f"  '{entity}' not found"); return
        vec = emb[name_to_idx[entity]:name_to_idx[entity]+1]
    else:
        vec = e.EmbedSet(entity, emb, temp)

    if vec.ndim == 1:
        vec = vec[np.newaxis, :]

    hop_witnesses = []
    for EmbR in EmbRs:
        vec, w = e.Join(vec, EmbR, temp=temp, semiring=semiring)
        hop_witnesses.append(w)

    scores, _ = e.Join(vec, emb.T, temp=temp, semiring=semiring)
    scores = scores[0]

    top_k_idx = np.argsort(scores)[::-1][:top_k]
    print(f"\nQ: {question} '{entity}'?")
    for i in top_k_idx:
        print(f"  {idx_to_name[i]:30s}  (score: {scores[i]:.4f})")

In [ ]:
query("What diseases linked via proteins to", "Donepezil", EmbR["drug_protein"], EmbR["disease_protein"])